### Imports

In [ ]:
import numpy as np
import itertools
from mogra.datatypes import Shruti, SSwar, SAPTAK_MARKS
from mogra.tonnetz import EFGenus, Tonnetz

### Toy Problem

I have a swarasamooha. I'm operating within a bounded tonnetz net.<br>
I want to evaluate a "cost" of each shruti assignment to every note in the samooha.

e.g.<br>
my samooha is `b1 = Sgn,Sn,SgSggn,n,n,S` and set of notes are `{S, g, n}` with options within my net `{S, g1/g2, n1/n2}`<br>
I'd like to arrive at a parameterization/family `J` that evaluate `J(b1(S,g1,n1)), J(b1(S,g1,n2)), J(b1(S,g2,n1)), J(b1(S,g2,n2))`

Eventually, I'd like to use this for contrastive learning:<br>
for 2 samoohas, `b1, b2` if I know for a fact that `{S, g1, n2}` minimizes `J(b1)` and `{S, g2, n1}` minimizes `J(b2)`,<br>
then I can use this fact to learn `J`

**The nature of J**

0th order
- collapse `b` into a histogram; `J` is not a function of a histogram on notes
- this ignores *movement/momentum*

aribitrary J
- ?

### Bgsr v/s Bpls

In [ ]:
primes = [3, 5]
ef = EFGenus(primes=primes, powers=[3, 1])
tn = Tonnetz(ef)

In [ ]:
# raag ground truths for this tonnetz
bgsr = [1, 4, 5, 7, 8, 10, 13]
bmpls = [7, 10, 12, 13, 15, 16, 19]

In [ ]:
# create "good" and "bad" samples given a samooha: list[SSwar]
# e.g.
b = ",n S g m P".split(" ")
b = [SSwar.from_string(s) for s in b]

In [ ]:
bad_paths = []
good_paths = []
def create_samples(samooha: list[SSwar], ground_truth: list[int]):
    ground_truth_coordinates = np.array(tn.node_coordinates)[bgsr]
    
    global bad_paths, good_paths
    # take a cartesian product of all options
    all_options = []
    for ss in samooha:
        all_options.append(tn.get_swar_options(ss.swar.name))
    all_paths = itertools.product(*all_options)
    
    for path in all_paths:
        # if all elements of the path are in tn.node_coordinates[ground_truth]:
        if all([cc in ground_truth_coordinates for cc in path]):
            # this is a good path
            good_paths.append(path)
        else:
            # this is a bad path
            bad_paths.append(path)

In [ ]:
good_paths

Harmono-Frequency Space Encoding

In [ ]:
def encode_hf(coordinate, saptak_mark: str):
    """
    TODO: replace the input with a Shruti object
    """
    if saptak_mark not in SAPTAK_MARKS:
        raise ValueError(f"Invalid saptak mark: {saptak_mark}. Must be one of {SAPTAK_MARKS}.")
    freq = float(tn.coord_to_ratio(coordinate))
    return (*coordinate, freq)